# Sistema Predictivo de Retención de Clientes

Este es un primer notebook para practicar el flujo completo con el dataset Telco Customer Churn. Primero lo cargo y reviso; después limpio algunos datos, pruebo tres modelos y hago una cuenta económica sencilla con supuestos editables.

La decisión económica es sólo una simulación. El dataset no trae información de campañas reales.


## 1. Preparar las librerías

Colab ya incluye pandas y scikit-learn. Instalo las librerías que uso para el balanceo, XGBoost y la conexión opcional con PostgreSQL.


In [ ]:
%pip -q install imbalanced-learn xgboost sqlalchemy psycopg2-binary


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbalancedPipeline
from xgboost import XGBClassifier

# Fijo una semilla para que la separación aleatoria se pueda repetir.
SEMILLA = 42


## 2. Cargar y revisar el dataset

Uso el mismo archivo indicado en el material de clase. No guardo el CSV en GitHub porque se puede volver a descargar desde este enlace.


In [ ]:
URL_DATASET = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# pandas lee el archivo directamente desde el enlace.
df_original = pd.read_csv(URL_DATASET)

print("Filas y columnas:", df_original.shape)
display(df_original.head())
print("\nColumnas:")
print(df_original.columns.tolist())


In [ ]:
# Compruebo que estén las columnas principales antes de seguir.
columnas_necesarias = ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
faltantes = [columna for columna in columnas_necesarias if columna not in df_original.columns]

if faltantes:
    raise ValueError(f"Faltan columnas que necesito: {faltantes}")

print("Valores vacíos por columna:")
display(df_original.isna().sum().sort_values(ascending=False).head(10))


## 3. Limpieza básica y primera exploración

`TotalCharges` puede llegar como texto y tiene algunos espacios vacíos. Lo convierto a número. Para clientes con cero meses de antigüedad y cargo total vacío, uso 0; los otros valores faltantes los completa el imputador durante el modelado.


In [ ]:
df = df_original.copy()

# Cambio el tipo de esta columna, porque como texto no se puede usar en cálculos.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# En el dataset, las filas sin TotalCharges y tenure igual a cero son clientes nuevos.
clientes_nuevos = (df["tenure"] == 0) & (df["TotalCharges"].isna())
df.loc[clientes_nuevos, "TotalCharges"] = 0

# Convierto la respuesta Yes/No a 1/0 para que el modelo la pueda aprender.
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# Quito filas sin identificador o sin respuesta; no servirían para entrenar.
df = df.dropna(subset=["customerID", "Churn"]).copy()

print("Filas después de la limpieza:", len(df))
print("Proporción de bajas:")
display(df["Churn"].value_counts(normalize=True).rename("proporción"))


In [ ]:
# Este gráfico ayuda a ver si la variable objetivo está desbalanceada.
df["Churn"].value_counts().sort_index().plot(kind="bar", color=["steelblue", "tomato"])
plt.xticks([0, 1], ["No se fue", "Se fue"], rotation=0)
plt.ylabel("Cantidad de clientes")
plt.title("Clientes por estado de churn")
plt.show()


## 4. Separar los datos y preparar las columnas

Dejo una parte para probar los modelos al final. Uso `stratify` para conservar una proporción parecida de bajas en entrenamiento y prueba. El balanceo queda dentro del pipeline y se aplica sólo al entrenamiento.


In [ ]:
# customerID identifica a cada persona pero no explica por sí solo si se va.
columnas_excluidas = ["Churn", "customerID"]
X = df.drop(columns=columnas_excluidas)
y = df["Churn"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEMILLA,
    stratify=y,
)

columnas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
columnas_texto = X.select_dtypes(exclude=["number"]).columns.tolist()

# A las columnas numéricas les completo vacíos con la mediana y las escalo.
preparar_numeros = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

# A las columnas de texto les completo vacíos con el valor más frecuente y
# convierto sus categorías a columnas numéricas.
preparar_texto = Pipeline(steps=[
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore")),
])

preparar_datos = ColumnTransformer(transformers=[
    ("numeros", preparar_numeros, columnas_numericas),
    ("texto", preparar_texto, columnas_texto),
])

print("Filas de entrenamiento:", len(X_train))
print("Filas de prueba:", len(X_test))


## 5. Entrenar y comparar modelos

Pruebo tres modelos pedidos para el proyecto: Regresión Logística, Random Forest y XGBoost. Uso `RandomOverSampler` dentro de cada pipeline para balancear sólo los datos que se usan para entrenar. Calibro sus probabilidades con una validación interna porque después las uso en un cálculo económico.


In [ ]:
# Calculo el peso de las bajas para que XGBoost también tenga en cuenta
# que hay menos casos positivos que negativos.
cantidad_no = int((y_train == 0).sum())
cantidad_si = int((y_train == 1).sum())
peso_bajas = cantidad_no / cantidad_si

modelos_base = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=SEMILLA),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        random_state=SEMILLA,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=peso_bajas,
        eval_metric="logloss",
        random_state=SEMILLA,
        n_jobs=-1,
    ),
}

modelos_entrenados = {}
resultados = []

for nombre, modelo in modelos_base.items():
    # El preprocesamiento y el balanceo se ajustan dentro de cada partición,
    # evitando que los datos de prueba pasen al entrenamiento.
    pipeline = ImbalancedPipeline(steps=[
        ("preparar", preparar_datos),
        ("balancear", RandomOverSampler(random_state=SEMILLA)),
        ("modelo", modelo),
    ])

    modelo_calibrado = CalibratedClassifierCV(
        estimator=pipeline,
        method="sigmoid",
        cv=3,
    )
    modelo_calibrado.fit(X_train, y_train)
    probabilidad = modelo_calibrado.predict_proba(X_test)[:, 1]
    prediccion = (probabilidad >= 0.5).astype(int)

    modelos_entrenados[nombre] = modelo_calibrado
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, prediccion),
        "precision": precision_score(y_test, prediccion, zero_division=0),
        "recall": recall_score(y_test, prediccion, zero_division=0),
        "f1": f1_score(y_test, prediccion, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilidad),
        "brier": brier_score_loss(y_test, probabilidad),
    })

tabla_resultados = pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)
display(tabla_resultados.round(3))


## 6. Probar una decisión económica

Elijo como modelo de referencia el que tuvo mayor ROC AUC en la prueba. Después uso supuestos editables para estimar un valor esperado por cliente. Estos números son sólo para mostrar cómo funcionaría el cálculo; no son cifras que entregue el dataset.


In [ ]:
mejor_nombre = tabla_resultados.iloc[0]["modelo"]
mejor_modelo = modelos_entrenados[mejor_nombre]

# Estos supuestos deberían reemplazarse por datos reales del negocio.
COSTO_CAMPANIA = 20.0
MESES_ESPERADOS = 6
MARGEN_ESTIMADO = 0.40
EXITO_CAMPANIA = 0.30

probabilidad_baja = mejor_modelo.predict_proba(X_test)[:, 1]
cargo_mensual = df.loc[X_test.index, "MonthlyCharges"].to_numpy()

# Aproximo el beneficio como margen esperado durante algunos meses.
beneficio_si_se_retiene = cargo_mensual * MESES_ESPERADOS * MARGEN_ESTIMADO
valor_esperado_neto = (
    probabilidad_baja * EXITO_CAMPANIA * beneficio_si_se_retiene
    - COSTO_CAMPANIA
)

# Si el valor esperado es positivo, la campaña parece conveniente bajo estos supuestos.
recomendar_campania = valor_esperado_neto > 0

print("Modelo elegido:", mejor_nombre)
print("Clientes de prueba:", len(X_test))
print("Clientes para contactar:", int(recomendar_campania.sum()))
print("Valor esperado neto total estimado: $", round(valor_esperado_neto.sum(), 2))


In [ ]:
# La matriz de confusión económica se presenta de forma simple.
# El valor esperado no demuestra que la campaña vaya a funcionar en la realidad.
matriz_decision = pd.DataFrame({
    "resultado_real": y_test.map({0: "No churn", 1: "Churn"}).to_numpy(),
    "recomendar_campania": recomendar_campania,
    "probabilidad_churn": probabilidad_baja,
    "valor_esperado_neto": valor_esperado_neto,
})

resumen_decision = matriz_decision.groupby(
    ["resultado_real", "recomendar_campania"],
    as_index=False,
).agg(
    clientes=("probabilidad_churn", "size"),
    probabilidad_promedio=("probabilidad_churn", "mean"),
    valor_esperado_total=("valor_esperado_neto", "sum"),
)
display(resumen_decision.round(2))


## 7. Exportar una tabla para revisar

Guardo solamente las predicciones del conjunto de prueba. Se puede importar este CSV en Power BI para practicar una visualización. No contiene resultados de campañas reales.


In [ ]:
predicciones = pd.DataFrame({
    "customer_id": df.loc[X_test.index, "customerID"].to_numpy(),
    "churn_real": y_test.to_numpy(),
    "probabilidad_churn": probabilidad_baja,
    "valor_esperado_neto": valor_esperado_neto,
    "recomendar_campania": recomendar_campania,
})

predicciones.to_csv("predicciones_para_powerbi.csv", index=False)
print("Se guardó predicciones_para_powerbi.csv")
display(predicciones.head())


## 8. Carga opcional en PostgreSQL

Esta parte queda desactivada por defecto. Primero hay que ejecutar el SQL del repositorio y configurar `DATABASE_URL` como secreto o variable de entorno en Colab. La celda vuelve a cargar únicamente las tablas de este proyecto, así que se debe usar una base de práctica.


In [ ]:
CARGAR_EN_POSTGRES = False

if CARGAR_EN_POSTGRES:
    from sqlalchemy import create_engine, text

    database_url = os.environ.get("DATABASE_URL")
    if not database_url:
        raise ValueError("Falta configurar DATABASE_URL")

    # Dejo los nombres de las columnas iguales a los del script SQL.
    datos_sql = df.rename(columns={
        "customerID": "customer_id",
        "SeniorCitizen": "senior_citizen",
        "PhoneService": "phone_service",
        "MultipleLines": "multiple_lines",
        "InternetService": "internet_service",
        "OnlineSecurity": "online_security",
        "OnlineBackup": "online_backup",
        "DeviceProtection": "device_protection",
        "TechSupport": "tech_support",
        "StreamingTV": "streaming_tv",
        "StreamingMovies": "streaming_movies",
        "Contract": "contract",
        "PaperlessBilling": "paperless_billing",
        "PaymentMethod": "payment_method",
        "MonthlyCharges": "monthly_charges",
        "TotalCharges": "total_charges",
        "Churn": "churn",
    })

    dim_customer = datos_sql[[
        "customer_id", "gender", "senior_citizen", "partner", "dependents"
    ]]
    fact_subscription = datos_sql[[
        "customer_id", "tenure", "phone_service", "multiple_lines",
        "internet_service", "online_security", "online_backup",
        "device_protection", "tech_support", "streaming_tv",
        "streaming_movies", "contract", "paperless_billing",
        "payment_method", "monthly_charges", "total_charges", "churn"
    ]]

    engine = create_engine(database_url)
    with engine.begin() as conexion:
        # Borro primero la tabla que tiene la clave foránea y después la dimensión.
        conexion.execute(text("DELETE FROM fact_subscription"))
        conexion.execute(text("DELETE FROM dim_customer"))
        dim_customer.to_sql("dim_customer", conexion, if_exists="append", index=False)
        fact_subscription.to_sql("fact_subscription", conexion, if_exists="append", index=False)

    print("La carga en PostgreSQL terminó.")
else:
    print("Carga apagada. Para usarla, configura DATABASE_URL y cambia el valor a True.")
